# Tuned Lens — calibrated layer-wise predictions

**Reference**: Belrose et al., *Eliciting Latent Predictions from Transformers with the Tuned Lens*, [arxiv:2303.08112](https://arxiv.org/abs/2303.08112) (v6, Nov 2025).

**Library**: [AlignmentResearch/tuned-lens](https://github.com/AlignmentResearch/tuned-lens) (MIT, 585 stars, last commit 2025-08-07).

## Tuned Lens vs Logit Lens (notebook 19)

| | Logit Lens | Tuned Lens |
|---|---|---|
| Method | `softmax(LN(h_l) @ W_U)` directly | `softmax(LN((W_l · h_l + b_l)) @ W_U)` |
| Training | None | Per-layer affine `(W_l, b_l)` trained to minimize `KL(lens_l || final_logits)` |
| Calibration | Biased toward later layers | Calibrated across all depths |
| Cost | Free | ~1 matmul + 2M tokens to fit |
| Faithfulness | Often misleading at early layers | Tracks actual computation trajectory |

The key insight: each layer's residual is *not* in the output-space basis. The tuned lens learns a per-layer affine correction that projects the partial residual into a space where the unembedding is meaningful. This gives a much more honest picture of what the model "thinks" at each layer.

### Pretrained lens availability (HF Space `AlignmentResearch/tuned-lens/lens`)

| Family | Available |
|---|---|
| GPT-2 (small/med/large/xl) | ✓ |
| Pythia (70m → 12b) | ✓ |
| GPT-Neo / GPT-NeoX-20B | ✓ |
| OPT (125m → 30b) | ✓ |
| Vicuna | ✓ |
| Llama-3-8B | ✓ |
| **Qwen / Qwen2 / Qwen3 / Qwen3.5 / Qwen3.6** | ✗ must fit fresh |
| **Gemma / Gemma2 / Gemma3 / Gemma4** | ✗ must fit fresh |
| **Mistral / Mixtral** | ✗ must fit fresh |
| **Phi-3 / Phi-4** | ✗ must fit fresh |

**Honest 2026 coverage note**: the published lens set is frozen at 2023-era models. Any modern architecture (Qwen3.x, Gemma3/4, Mistral, Phi-4) must be fit from scratch — the cell 8 training loop handles this automatically.

In [ ]:
!pip install -q -U transformers accelerate safetensors huggingface_hub tuned-lens matplotlib tqdm datasets

## Config

Defaults chosen for **Colab T4 free tier**: Pythia-160m has a pretrained lens on the HF Space and runs end-to-end in <5 min. To fit fresh on a new arch, swap `MODEL_ID` — the notebook will detect the missing lens and fall through to training.

In [ ]:
import torch, os, json, math
from pathlib import Path

# --- swap this line to try another model ---
MODEL_ID = 'EleutherAI/pythia-160m'    # has pretrained lens + fits T4 free
# MODEL_ID = 'gpt2'                    # has pretrained lens
# MODEL_ID = 'meta-llama/Llama-3-8b'   # has pretrained lens (needs A100)
# MODEL_ID = 'Qwen/Qwen2.5-0.5B'       # no pretrained lens — triggers fresh fit

PROMPT          = 'The Eiffel Tower is located in the city of'
TOP_K           = 5
PRETRAINED_FIRST = True   # attempt download before fitting

# Fresh-fit hyperparams (only used if no pretrained lens found)
FIT_STEPS       = 200
FIT_BATCH       = 8
FIT_SEQLEN      = 256
FIT_LR          = 1e-3
FIT_TOKENS      = FIT_STEPS * FIT_BATCH * FIT_SEQLEN   # ~400k by default — bump to 2M for quality

OUT_DIR = Path('./tuned_lens_out')
OUT_DIR.mkdir(exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DTYPE  = torch.bfloat16 if DEVICE == 'cuda' else torch.float32
print(f'device={DEVICE} dtype={DTYPE} model={MODEL_ID}')

## Step 1 — load model and try pretrained lens

`TunedLens.from_model_and_pretrained(model)` queries the `AlignmentResearch/tuned-lens` HF Space for a checkpoint matching the model's config. If none exists, we catch the exception and fall through to the fresh-fit path.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from tuned_lens import TunedLens

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=DTYPE,
    device_map=DEVICE,
    attn_implementation='sdpa',
)
model.eval()
tok = AutoTokenizer.from_pretrained(MODEL_ID)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

FRESH_TRAIN = True
lens = None

if PRETRAINED_FIRST:
    try:
        lens = TunedLens.from_model_and_pretrained(model).to(DEVICE)
        print(f'[OK] loaded pretrained tuned lens for {MODEL_ID}')
        FRESH_TRAIN = False
    except Exception as e:
        print(f'[miss] no pretrained lens for {MODEL_ID} — will fit fresh')
        print(f'       reason: {type(e).__name__}: {str(e)[:200]}')
        FRESH_TRAIN = True

print(f'FRESH_TRAIN = {FRESH_TRAIN}')

## Step 2 — fresh-fit (only if no pretrained lens)

We train the per-layer affine on streamed Pile samples. Objective: `KL(tuned_lens_logits_l || final_logits)` per layer. AdamW, lr=1e-3, ~400k tokens default (bump `FIT_TOKENS` for quality — the paper uses 2B).

In [ ]:
if FRESH_TRAIN:
    from tuned_lens import TunedLens
    from datasets import load_dataset
    from torch.optim import AdamW
    from tqdm.auto import tqdm
    import torch.nn.functional as F

    # fresh lens: per-layer affine (one per hidden state before block L)
    lens = TunedLens.from_model(model).to(DEVICE)

    # the affine maps are the only trainable params
    for p in model.parameters():
        p.requires_grad_(False)
    trainable = [p for p in lens.parameters() if p.requires_grad]
    n_params = sum(p.numel() for p in trainable)
    print(f'training {n_params/1e6:.2f}M lens params over ~{FIT_TOKENS/1e6:.2f}M tokens')

    opt = AdamW(trainable, lr=FIT_LR, betas=(0.9, 0.999), weight_decay=1e-3)

    # stream the Pile (monology copy is public)
    ds = load_dataset('monology/pile-uncopyrighted', split='train', streaming=True)
    it = iter(ds)

    def next_batch():
        texts = []
        while len(texts) < FIT_BATCH:
            try:
                ex = next(it)
            except StopIteration:
                break
            if ex.get('text'):
                texts.append(ex['text'])
        enc = tok(texts, return_tensors='pt', max_length=FIT_SEQLEN,
                  truncation=True, padding='max_length')
        return {k: v.to(DEVICE) for k, v in enc.items()}

    model.eval()
    pbar = tqdm(range(FIT_STEPS), desc='fit tuned lens')
    for step in pbar:
        batch = next_batch()
        with torch.no_grad():
            out = model(**batch, output_hidden_states=True, use_cache=False)
            final_logits = out.logits                 # [B, T, V]
            final_logp   = F.log_softmax(final_logits.float(), dim=-1)
            hidden       = out.hidden_states          # tuple of L+1, each [B, T, D]

        loss = 0.0
        for l in range(len(hidden) - 1):              # one per block output
            h = hidden[l]
            logits_l = lens.forward(h, l)             # [B, T, V]
            logp_l   = F.log_softmax(logits_l.float(), dim=-1)
            # KL(tuned_l || final) = sum exp(tuned) * (tuned - final)
            loss = loss + F.kl_div(logp_l, final_logp,
                                   reduction='batchmean', log_target=True)
        loss = loss / max(1, len(hidden) - 1)

        opt.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(trainable, 1.0)
        opt.step()

        if step % 10 == 0:
            pbar.set_postfix(kl=f'{loss.item():.4f}')

    # save locally
    lens_path = OUT_DIR / 'fitted_lens'
    lens_path.mkdir(exist_ok=True)
    lens.save(str(lens_path))
    print(f'[OK] fitted lens saved to {lens_path}')
else:
    print('skipping fresh fit — using pretrained lens')

## Step 3 — apply lens + raw Logit Lens on a prompt

For each layer we compute both the **raw logit-lens** (direct `lm_head(LN(h))`) and the **tuned lens** (`lens.forward(h, l)`). The contrast shows why raw logit lens is misleading at early layers: without calibration, early residuals decode to "typical output" tokens rather than reflecting the partial computation.

In [ ]:
import torch.nn.functional as F

# locate the final layernorm + unembedding for raw logit lens
base = getattr(model, 'transformer', None) or getattr(model, 'model', None) or getattr(model, 'gpt_neox', None)
final_ln = None
for name in ('ln_f', 'final_layer_norm', 'norm'):
    if hasattr(base, name):
        final_ln = getattr(base, name)
        break
if final_ln is None:
    final_ln = torch.nn.Identity()
unembed = model.get_output_embeddings()

enc = tok(PROMPT, return_tensors='pt').to(DEVICE)
with torch.no_grad():
    out = model(**enc, output_hidden_states=True, use_cache=False)
    hidden = out.hidden_states
    final_logits = out.logits[:, -1, :].float()
    final_probs  = F.softmax(final_logits, dim=-1)
    final_top    = final_probs.topk(TOP_K, dim=-1)

print(f'\nPROMPT: {PROMPT!r}')
print(f'FINAL top-{TOP_K}: ' + ', '.join(
    f'{tok.decode([t])!r}={p:.3f}'
    for t, p in zip(final_top.indices[0].tolist(), final_top.values[0].tolist())))

n_layers = len(hidden) - 1
raw_topk   = []
tuned_topk = []
kl_raw, kl_tuned = [], []
final_logp = F.log_softmax(final_logits, dim=-1)

print(f'\n{"L":>3} | {"raw logit-lens top-1":<28} | {"tuned lens top-1":<28} | KL(raw) | KL(tuned)')
print('-' * 110)
for l in range(n_layers):
    h_last = hidden[l][:, -1, :]

    # raw logit lens
    with torch.no_grad():
        raw_logits = unembed(final_ln(h_last)).float()
        raw_logp   = F.log_softmax(raw_logits, dim=-1)
        raw_p      = raw_logp.exp()
        raw_top    = raw_p.topk(TOP_K, dim=-1)

        # tuned lens — forward takes full hidden, we index last pos
        t_logits = lens.forward(hidden[l], l)[:, -1, :].float()
        t_logp   = F.log_softmax(t_logits, dim=-1)
        t_p      = t_logp.exp()
        t_top    = t_p.topk(TOP_K, dim=-1)

        kl_r = F.kl_div(raw_logp, final_logp, reduction='sum', log_target=True).item()
        kl_t = F.kl_div(t_logp,   final_logp, reduction='sum', log_target=True).item()

    raw_topk.append((raw_top.indices[0].tolist(), raw_top.values[0].tolist()))
    tuned_topk.append((t_top.indices[0].tolist(), t_top.values[0].tolist()))
    kl_raw.append(kl_r)
    kl_tuned.append(kl_t)

    raw_str   = f'{tok.decode([raw_top.indices[0,0].item()])!r}={raw_top.values[0,0].item():.2f}'
    tuned_str = f'{tok.decode([t_top.indices[0,0].item()])!r}={t_top.values[0,0].item():.2f}'
    print(f'{l:>3} | {raw_str:<28} | {tuned_str:<28} | {kl_r:7.3f} | {kl_t:7.3f}')

## Step 4 — visualize: side-by-side heatmap + KL divergence curve

Left panel: raw logit-lens top-k probabilities per layer.  Middle panel: tuned-lens top-k.  Right panel: KL divergence vs final logits — tuned lens should be uniformly lower than raw logit lens across depth.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# collect per-layer top-K probability matrices (rows=layer, cols=rank)
raw_prob_mat   = np.array([p for _, p in raw_topk])     # [L, K]
tuned_prob_mat = np.array([p for _, p in tuned_topk])

# label grid uses the tuned-lens top tokens (more meaningful)
raw_labels   = [[tok.decode([t]).replace('\n', '\\n')[:10] for t in ids] for ids, _ in raw_topk]
tuned_labels = [[tok.decode([t]).replace('\n', '\\n')[:10] for t in ids] for ids, _ in tuned_topk]

fig, axes = plt.subplots(1, 3, figsize=(20, max(6, n_layers * 0.3)),
                        gridspec_kw={'width_ratios': [2.0, 2.0, 1.0]})

def plot_heat(ax, mat, labels, title):
    im = ax.imshow(mat, aspect='auto', cmap='viridis', vmin=0, vmax=1)
    ax.set_title(title)
    ax.set_xlabel('top-K rank')
    ax.set_ylabel('layer')
    ax.set_xticks(range(TOP_K))
    ax.set_yticks(range(n_layers))
    for i in range(n_layers):
        for k in range(TOP_K):
            ax.text(k, i, labels[i][k], ha='center', va='center',
                    color='white' if mat[i, k] < 0.5 else 'black', fontsize=7)
    plt.colorbar(im, ax=ax, fraction=0.046)

plot_heat(axes[0], raw_prob_mat,   raw_labels,   f'Raw Logit Lens — {MODEL_ID}')
plot_heat(axes[1], tuned_prob_mat, tuned_labels, f'Tuned Lens — {MODEL_ID}')

ax = axes[2]
layers = np.arange(n_layers)
ax.plot(kl_raw,   layers, 'o-', label='raw logit lens', color='C3')
ax.plot(kl_tuned, layers, 's-', label='tuned lens',     color='C0')
ax.invert_yaxis()
ax.set_xlabel('KL vs final logits (nats)')
ax.set_ylabel('layer')
ax.set_title('Calibration (lower = closer to final)')
ax.set_xscale('symlog')
ax.grid(alpha=0.3)
ax.legend()

plt.suptitle(f'Logit Lens vs Tuned Lens — prompt: {PROMPT!r}', y=1.02)
plt.tight_layout()
fig.savefig(OUT_DIR / 'tuned_lens_comparison.png', dpi=140, bbox_inches='tight')
print(f'[saved] {OUT_DIR}/tuned_lens_comparison.png')
plt.show()

## Step 5 — save + upload (only if we fit fresh)

If we trained a fresh lens, publish it so others skip the 15-20 min fit. If we loaded pretrained, just record the source for provenance.

In [ ]:
from huggingface_hub import HfApi, create_repo

if FRESH_TRAIN:
    user = os.environ.get('HF_USER', 'your-username')
    model_slug = MODEL_ID.split('/')[-1].lower()
    repo_id = f'{user}/{model_slug}-tuned-lens'
    print(f'upload target: {repo_id}')
    print('set HF_USER env var + `huggingface-cli login`, then uncomment:')
    print()
    print('# create_repo(repo_id, exist_ok=True)')
    print(f"# HfApi().upload_folder(folder_path='{OUT_DIR}/fitted_lens', repo_id=repo_id)")

    # write a minimal provenance card
    card = f'''---
base_model: {MODEL_ID}
tags: [tuned-lens, interpretability, mechanistic-interpretability]
license: mit
---
# Tuned Lens for `{MODEL_ID}`

Fitted per-layer affine (Belrose et al. 2023, arxiv:2303.08112) trained to minimize
KL against the final logits over ~{FIT_TOKENS/1e6:.1f}M Pile tokens.

- **Steps**: {FIT_STEPS} at lr={FIT_LR}, batch={FIT_BATCH}, seq_len={FIT_SEQLEN}
- **Loss**: mean KL across layers
- **Layers**: {n_layers}

## Usage
```python
from tuned_lens import TunedLens
from transformers import AutoModelForCausalLM
model = AutoModelForCausalLM.from_pretrained("{MODEL_ID}")
lens  = TunedLens.from_model_and_pretrained(model, lens_resource_id="{repo_id}")
```
'''
    (OUT_DIR / 'fitted_lens' / 'README.md').write_text(card)
    print('[OK] model card written')
else:
    src = f'huggingface.co/spaces/AlignmentResearch/tuned-lens/tree/main/lens/{MODEL_ID}'
    print(f'used pretrained lens from: {src}')
    print('no upload needed — credit AlignmentResearch if you re-share.')

print('\ndone.')